In [ ]:
import os, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch :", torch.__version__)
print("Appareil utilisé :", DEVICE) 

In [21]:
DATA_DIR = Path("data/spectrograms/")   

IMG_SIZE    = 128    
BATCH_SIZE  = 32
NUM_EPOCHS  = 30
LR          = 1e-3
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
SEED        = 42

USE_SPEAKER_SPLIT = True   # True = séparation par locuteur // False = séparation aléatoire.
USE_SPECAUGMENT   = True   # cf diapo p.4
USE_CLASS_WEIGHTS = True   # compense le fait que certaines émotions ont plus d'exemples que d'autres

seed_everything(SEED)

In [ ]:
probe = datasets.ImageFolder(str(DATA_DIR))

classes = probe.classes
n_classes = len(classes)
print(f"{n_classes} émotions détectées :", classes, "\n")

from collections import Counter
counts = Counter([classes[label] for _, label in probe.samples])
for emo in classes:
    print(f"  {emo:12s} : {counts[emo]} images")
print(f"\n  TOTAL : {len(probe.samples)} images")

Peut etre a dégager en fonction de ce que fait baptiste 

In [ ]:
def split_by_speaker(samples, val_ratio, test_ratio, seed=42):
    '''Découpe par locuteur : un acteur entier va dans un seul split.'''
    rng = random.Random(seed)
    # indices des images regroupés par acteur
    by_spk = {}
    for idx, (path, _) in enumerate(samples):
        by_spk.setdefault(speaker_of(path), []).append(idx)

    spks = list(by_spk.keys())
    rng.shuffle(spks)
    n = len(spks)
    n_test = max(1, round(n * test_ratio))
    n_val  = max(1, round(n * val_ratio))

    test_spk  = spks[:n_test]
    val_spk   = spks[n_test:n_test + n_val]
    train_spk = spks[n_test + n_val:]

    def gather(spk_list):
        out = []
        for s in spk_list:
            out += by_spk[s]
        return out

    print("Acteurs -> train:", train_spk, "| val:", val_spk, "| test:", test_spk)
    return gather(train_spk), gather(val_spk), gather(test_spk)

def split_random(samples, val_ratio, test_ratio, seed=42):
    '''Découpe aléatoire (à éviter pour la note finale, sert juste de comparaison).'''
    rng = random.Random(seed)
    idx = list(range(len(samples)))
    rng.shuffle(idx)
    n = len(idx)
    n_test = round(n * test_ratio)
    n_val  = round(n * val_ratio)
    test_idx  = idx[:n_test]
    val_idx   = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]
    return train_idx, val_idx, test_idx

if USE_SPEAKER_SPLIT:
    train_idx, val_idx, test_idx = split_by_speaker(probe.samples, VAL_RATIO, TEST_RATIO, SEED)
else:
    train_idx, val_idx, test_idx = split_random(probe.samples, VAL_RATIO, TEST_RATIO, SEED)

print(f"\nTailles -> train: {len(train_idx)} | val: {len(val_idx)} | test: {len(test_idx)} images")

Definition du réseau 

In [ ]:
class ConvBlock(nn.Module):
    '''Deux convolutions 3x3 (chacune suivie de BatchNorm + ReLU) puis un MaxPool 2x2.'''
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, kernel_size=3, padding=1) 
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.pool  = nn.MaxPool2d(2)                                      

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        return self.pool(x)


class EmotionCNN(nn.Module):
    def __init__(self, n_classes, in_ch=1):
        super().__init__()
        self.block1 = ConvBlock(in_ch, 32)
        self.block2 = ConvBlock(32, 64)
        self.block3 = ConvBlock(64, 128)
        self.block4 = ConvBlock(128, 256)
        self.gap     = nn.AdaptiveAvgPool2d(1)   
        self.dropout = nn.Dropout(0.5)
        self.fc      = nn.Linear(256, n_classes) 

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x)            
        x = torch.flatten(x, 1)    
        x = self.dropout(x)
        return self.fc(x)          


model = EmotionCNN(n_classes=n_classes, in_ch=1).to(DEVICE)
print(model)

In [ ]:
# Poids de classes = inverse de la fréquence dans le train
if USE_CLASS_WEIGHTS:
    train_labels = [full_train.samples[i][1] for i in train_idx]
    class_count = np.bincount(train_labels, minlength=n_classes)
    weights = class_count.sum() / (n_classes * np.maximum(class_count, 1))
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print("Poids par classe :", {classes[i]: round(float(w), 2) for i, w in enumerate(class_weights)})
else:
    class_weights = None

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
# Réduit le learning rate quand la validation stagne (aide à mieux converger)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

In [27]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = criterion(logits, y)

            if is_train:
                optimizer.zero_grad()   # remet les gradients à zéro
                loss.backward()         # calcule les gradients
                optimizer.step()        # met à jour les poids

            total_loss += loss.item() * x.size(0)
            correct    += (logits.argmax(1) == y).sum().item()
            total      += x.size(0)

    return total_loss / total, correct / total

In [28]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_acc = run_epoch(model, val_loader,   criterion, optimizer=None)
    scheduler.step(va_acc)

    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(va_acc)

    flag = ""
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        flag = "  <-- meilleur"

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f}{flag}")

print(f"\nMeilleure accuracy de validation : {best_val_acc:.3f}")
# On recharge les poids du meilleur modèle
if best_state is not None:
    model.load_state_dict(best_state)

NameError: name 'train_loader' is not defined